# LEAR — model selection

Forecasts are saved under `results/lear/<split>/<run>/<model>.csv`:

- `val/screening` — one week from each validation month
- `val/full` — every day of the validation set
- `test/full` — backtest on the test set

LEAR is the LASSO-estimated autoregressive model from `epftoolbox`. Every day it is recalibrated on the last `calibration_window` days and forecasts all 24 hourly prices of the next day, with one LASSO regression per hour. The inputs are the prices of days D-1, D-2, D-3 and D-7, the two exogenous columns from `EPF_EXOG` (`PL_gen_fc_balance` and `PL_load_fc`) for days D, D-1 and D-7, and day-of-week dummies. Both columns are built from the forecasts in the `epf` dataset (see `01_merging.ipynb`).

Three calibration windows are compared: 364, 728 and 1092 days, that is 1, 2 and 3 years of 364 days. Each is a whole number of weeks, so the window always starts on the same day of the week.

The model needs history before every validation day, so the train and validation sets are joined into `dataset_full`. The training set is exactly 1092 days long, so even the first validation day has a full window for the longest setting.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

import pandas as pd

from src.config import EPF_EXOG
from src.utils.epf import evaluate_lear, forecast_path
from src.utils.evaluation import compare_models, load_forecasts
from src.utils.file_io import load_final_dataset

/opt/homebrew/Caskroom/miniforge/base/envs/epf-lear-dnn/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
dataset_train = load_final_dataset("epf", "pl", "train")
dataset_val = load_final_dataset("epf", "pl", "val")
dataset_full = pd.concat([dataset_train, dataset_val])
prices = dataset_full["PL_price"]

configs = [
    {"model": "LEAR_364", "calibration_window": 364, "exog": EPF_EXOG},
    {"model": "LEAR_728", "calibration_window": 728, "exog": EPF_EXOG},
    {"model": "LEAR_1092", "calibration_window": 1092, "exog": EPF_EXOG},
]

val_start = dataset_val.index.min()
val_end = dataset_val.index.max()
month_starts = list(pd.date_range(val_start, val_end, freq="MS"))

## 1. Full validation

Each window is run once over every day of the validation set (126 days). The forecasts are saved to `results/lear/val/full/<model>.csv`, and the comparison below reads the files instead of recomputing them.

In [ ]:
for config in configs:
    evaluate_lear(
        dataset=dataset_full,
        calibration_window=config["calibration_window"],
        exog=config["exog"],
        forecast_start=val_start,
        forecast_end=val_end,
        path=forecast_path(config["model"], split="val", run="full"),
    )

## 2. Full validation comparison

Forecasts are scored against `PL_price`. MAE and RMSE are in EUR/MWh. rMAE is the MAE divided by the MAE of a naive forecast that repeats the price from the same hour one week earlier, so a value below 1 beats that naive forecast. Lower is better for all three.

In [3]:
full_models = [config["model"] for config in configs]
forecasts = load_forecasts(
    [forecast_path(model, split="val", run="full") for model in full_models]
)

compare_models(forecasts, prices, metrics=["rMAE", "MAE", "RMSE"])

,rMAE,MAE,RMSE
model,,,
LEAR_364,0.684534,18.531291,29.753640
LEAR_728,0.671208,18.170537,28.764687
LEAR_1092,0.672119,18.195204,28.902790


## 3. Conclusions

`LEAR_728` is the best window on all three metrics (rMAE 0.671, MAE 18.2, RMSE 28.8), and `LEAR_1092` is almost the same. `LEAR_364` is worse on every metric, so one year of history is too short, while a third year does not improve on two. All three windows beat the weekly naive forecast.

Whether these gaps are significant, and how LEAR compares with the DNN, is checked in `03_LEAR_DNN_comparison.ipynb`.